# 04b — LoRA persona DPO: gentler variants (Task 5 comparison)

Accelerator: **GPU T4 x2** (one GPU is used), Internet **on**. No inputs needed. Expected time: ~10–15 min.

The first LoRA-DPO run (`configs/lora_dpo.yaml`: 3 epochs, beta 0.1) saturated held-out reward accuracy by
step 10 and drifted into generic rudeness. These two variants change one setting each:

| Variant | Change |
|---|---|
| `lora_dpo_1ep` | 1 epoch instead of 3 (tests over-training) |
| `lora_dpo_beta05` | beta 0.5 instead of 0.1 (stronger pull towards the reference model) |

Same data split, LoRA config, and decoding settings as the first run. When done: **Save Version**.

In [ ]:
REPO_URL = "https://github.com/madgod2004/small-model-alignment-starter.git"
REPO_DIR = "/kaggle/working/small-model-alignment-starter"
BRANCH = "main"

In [ ]:
import os
if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}
!git log --oneline -1

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors matplotlib "peft>=0.11.0" "trl>=0.12.0" pytest
!python -m pip install -q -e . --no-deps

In [ ]:
!nvidia-smi
!python scripts/check_env.py --expect_gpus 2

## 1. Train both variants

In [ ]:
VARIANTS = ["lora_dpo_1ep", "lora_dpo_beta05"]
!mkdir -p outputs/logs
for variant in VARIANTS:
    !PYTHONUNBUFFERED=1 CUDA_VISIBLE_DEVICES=0 python scripts/train_lora_dpo.py --config configs/{variant}.yaml 2>&1 | grep -v -i warn | tee outputs/logs/train_{variant}.log

In [ ]:
import json
for variant in VARIANTS:
    summary = json.load(open(f"outputs/{variant}/summary.json"))
    e = summary["final_eval"]
    print(f"{variant}: {summary['optimizer_steps']} steps, held-out loss {e['eval_loss']:.4f}, "
          f"acc {e['eval_rewards/accuracies']:.3f}, margin {e['eval_rewards/margins']:.3f}")

## 2. Generate on the style-evaluation prompts (same settings as the first run)

In [ ]:
for variant in VARIANTS:
    !python scripts/generate.py --model_path outputs/{variant}/merged --generator smollm2-360m-{variant.replace("_", "-")} --eval_set eval_sets/persona_style_eval.json --output_path outputs/generations/{variant}_persona_eval.json --prompt_format chat_template --overwrite 2>&1 | grep -v -i warn

In [ ]:
outputs = {v: json.load(open(f"outputs/generations/{v}_persona_eval.json")) for v in VARIANTS}
for i, row in enumerate(outputs[VARIANTS[0]]):
    print("=" * 100)
    print(f"[{row['category']}] {row['instruction']}")
    for v in VARIANTS:
        print(f"--- {v}:", outputs[v][i]["output"][:500])

## 3. Archive

In [ ]:
!zip -qr /kaggle/working/lora_dpo_variants_outputs.zip outputs/lora_dpo_1ep/*.json* outputs/lora_dpo_beta05/*.json* outputs/logs/train_lora_dpo_*.log outputs/generations/lora_dpo_*_persona_eval*.json
!ls -lh /kaggle/working/*.zip && du -sh outputs/lora_dpo_*/merged